## Setup

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "transfermarkt"

def silver(t):
    return spark.table(f"workspace.silver.{t}")

players = silver("players")
clubs = silver("clubs")
competitions = silver("competitions")
games = silver("games")
appearances = silver("appearances")
valuations = silver("player_valuations")
transfers = silver("transfers")

## Schemas we haven't inspected yet
`clubs`, `competitions`, and `appearances` were checked earlier. This fills in everything else — `transfers` matters most right now, since we need its real column names for the referential-integrity and fee checks below.

In [0]:
for t in ["transfers", "club_games", "game_events", "game_lineups", "countries", "national_teams"]:
    print(f"--- {t} ---")
    silver(t).printSchema()

## Referential integrity
An anti-join returns rows on the left with no match on the right — exactly what's needed to find a foreign key pointing at nothing. Zero here means every fact row's dimension lookups will actually resolve in Gold.

In [0]:
def orphan_count(fact_df, fact_col, dim_df, dim_col):
    return (fact_df.select(fact_col).distinct()
            .join(dim_df.select(dim_col), fact_df[fact_col] == dim_df[dim_col], "left_anti")
            .count())

checks = [
    ("appearances.player_id -> players", appearances, "player_id", players, "player_id"),
    ("appearances.game_id -> games", appearances, "game_id", games, "game_id"),
    ("appearances.competition_id -> competitions", appearances, "competition_id", competitions, "competition_id"),
    ("appearances.player_club_id -> clubs", appearances, "player_club_id", clubs, "club_id"),
    ("games.competition_id -> competitions", games, "competition_id", competitions, "competition_id"),
    ("games.home_club_id -> clubs", games, "home_club_id", clubs, "club_id"),
    ("games.away_club_id -> clubs", games, "away_club_id", clubs, "club_id"),
    ("valuations.player_id -> players", valuations, "player_id", players, "player_id"),
    ("valuations.current_club_id -> clubs", valuations, "current_club_id", clubs, "club_id"),
]

for label, fact_df, fact_col, dim_df, dim_col in checks:
    print(f"{label}: {orphan_count(fact_df, fact_col, dim_df, dim_col)} orphans")

## Referential integrity — transfers
Column names here (`from_club_id`, `to_club_id`) are my best guess for this dataset — swap in whatever Cell 2's printSchema actually showed before running this.

In [0]:
transfer_checks = [
    ("transfers.player_id -> players", transfers, "player_id", players, "player_id"),
    ("transfers.from_club_id -> clubs", transfers, "from_club_id", clubs, "club_id"),
    ("transfers.to_club_id -> clubs", transfers, "to_club_id", clubs, "club_id"),
]
for label, fact_df, fact_col, dim_df, dim_col in transfer_checks:
    print(f"{label}: {orphan_count(fact_df, fact_col, dim_df, dim_col)} orphans")

## Duplicate checks
Silver already deduped `players` and `games` on their real keys. This confirms it held, and checks the tables that weren't explicitly deduped.

In [0]:
print("Duplicate game_id:", games.groupBy("game_id").count().filter("count > 1").count())
print("Duplicate player_id:", players.groupBy("player_id").count().filter("count > 1").count())
print("Duplicate club_id:", clubs.groupBy("club_id").count().filter("count > 1").count())
print("Duplicate competition_id:", competitions.groupBy("competition_id").count().filter("count > 1").count())
print("Duplicate appearance_id:", appearances.groupBy("appearance_id").count().filter("count > 1").count())

## Missingness, column by column
`.show(vertical=True)` prints one column per line instead of a wide row — much more readable once a table has more than about 6 columns, which every table here does.

In [0]:
def null_report(df, name):
    total = df.count()
    print(f"--- {name} ({total} rows) ---")
    df.select([
        (F.count(F.when(F.col(c).isNull(), c)) / total * 100).alias(c)
        for c in df.columns
    ]).show(truncate=False, vertical=True)

for name, df in [("players", players), ("games", games), ("appearances", appearances),
                  ("player_valuations", valuations)]:
    null_report(df, name)

## Distributions and date coverage
Sanity-checks the actual value ranges (no negative goals, no absurd market values) and confirms what date range the data really covers, rather than trusting the dataset description.

In [0]:
players.select("market_value_in_eur", "height_in_cm", "international_caps").summary().show()
appearances.select("goals", "assists", "minutes_played").summary().show()

games.agg(F.min("date").alias("min_date"), F.max("date").alias("max_date")).show()
appearances.agg(F.min("date").alias("min_date"), F.max("date").alias("max_date")).show()
valuations.agg(F.min("date").alias("min_date"), F.max("date").alias("max_date")).show()

## Does the data make football sense?
Home win rate should land somewhere in the mid-40s percent — that's the well-known home-advantage baseline in football. Wildly off from that would suggest a processing error upstream, not a modeling choice.

In [0]:
results = games.withColumn(
    "result",
    F.when(F.col("home_club_goals") > F.col("away_club_goals"), "home_win")
     .when(F.col("home_club_goals") < F.col("away_club_goals"), "away_win")
     .otherwise("draw")
)
results.groupBy("result").count().withColumn(
    "pct", F.round(F.col("count") / games.count() * 100, 1)
).show()

## Transfers: profiling for the loan/free-transfer decision
Loans and free transfers typically show up as a zero or null fee — but "zero" could also mean "fee genuinely undisclosed," which isn't the same thing and shouldn't be silently lumped in as free. Eyeball the sample rows before trusting the count. Adjust "transfer_fee" if Cell 2 showed a different column name.

In [0]:
transfers.select("transfer_fee").summary().show()

zero_or_null = transfers.filter((F.col("transfer_fee") == 0) | F.col("transfer_fee").isNull())
print("Transfers with zero/null fee:", zero_or_null.count(), "out of", transfers.count())
zero_or_null.show(10)

## Transfers: testing a three-way transfer_type classification
The zero/null-fee group isn't one thing — it mixes genuine free transfers/loans with academy-to-senior-squad promotions recorded as "transfers" between two IDs that are really the same club. This tests a name-matching heuristic to separate the internal-squad-move case out, and prints the leftover free_or_loan sample so we can eyeball whether it's still catching things it shouldn't.

In [0]:
def strip_youth_suffix(col):
    return F.trim(F.regexp_replace(
        col, r"\s+(U1[0-9]|U2[0-9]|II|III|B|Youth|Yth\.?)$", ""
    ))

transfers_typed = (
    transfers
    .withColumn("from_club_root", strip_youth_suffix(F.col("from_club_name")))
    .withColumn("to_club_root", strip_youth_suffix(F.col("to_club_name")))
    .withColumn(
        "transfer_type",
        F.when(F.col("transfer_fee") > 0, "paid")
         .when(F.col("from_club_root") == F.col("to_club_root"), "internal_squad_move")
         .otherwise("free_or_loan")
    )
)

transfers_typed.groupBy("transfer_type").count().show()
transfers_typed.filter(F.col("transfer_type") == "free_or_loan").select(
    "from_club_name", "to_club_name", "transfer_fee"
).show(15, truncate=False)